In [0]:
from pyspark.sql.functions import col, count, when, lit

# Compute per-company data volume
company_confidence = (
    spark.read.table("bdc_share_cash_flow.cashflow.cashflow")
        .groupBy("CompanyCode")
        .agg(count("*").alias("record_count"))
        .withColumn("confidence",
            when(col("record_count") < 100, lit("LOW"))
            .when(col("record_count") < 1000, lit("MEDIUM"))
            .otherwise(lit("HIGH"))
        )
        .orderBy(col("record_count").desc())
)

display(company_confidence)
print("\nGuardrail: aggregations from LOW confidence companies must be flagged in agent output")

CompanyCode,record_count,confidence
1710,808423,HIGH
1010,107630,HIGH
AUC1,17054,HIGH
USC1,16921,HIGH
DEC1,16454,HIGH
1110,623,MEDIUM
3010,242,MEDIUM
L100,81,LOW
R300,51,LOW
R100,41,LOW



Guardrail: aggregations from LOW confidence companies must be flagged in agent output


In [0]:
import re

FORBIDDEN_ASSERTIONS = [
    r"\bconfirmed\s+fraud\b",
    r"\bdefinitely\s+fraud\b",
    r"\bis\s+definitely\s+illegal\b",
    r"\bmoney\s+laundering\s+is\s+occurring\b",
]

REQUIRED_HEDGE_TERMS = ["may", "might", "possibly", "could", "suggests", "indicates"]

def output_guardrail(text: str) -> dict:
    """Enforce honesty contract: no fabricated certainty."""
    violations = []
    for pattern in FORBIDDEN_ASSERTIONS:
        if re.search(pattern, text, re.IGNORECASE):
            violations.append(f"Forbidden assertion: {pattern}")
    
    has_hedge = any(term in text.lower() for term in REQUIRED_HEDGE_TERMS)
    
    return {
        "text": text,
        "violations": violations,
        "has_hedge": has_hedge,
        "safe": len(violations) == 0
    }

# Test against actual outputs from earlier iterations
test_v1_bad = "This is money laundering is occurring in the finance system."
test_v3_good = "This anomaly may indicate a large payment or year-end adjustment."

print("V1 output check:", output_guardrail(test_v1_bad))
print("\nV3 output check:", output_guardrail(test_v3_good))

V1 output check: {'text': 'This is money laundering is occurring in the finance system.', 'violations': ['Forbidden assertion: \\bmoney\\s+laundering\\s+is\\s+occurring\\b'], 'has_hedge': False, 'safe': False}

V3 output check: {'text': 'This anomaly may indicate a large payment or year-end adjustment.', 'violations': [], 'has_hedge': True, 'safe': True}


In [0]:
def confidence_aware_answer(company_code: str, question: str, agent_result: str) -> str:
    """Wrap agent answer with confidence flag from data guard."""
    confidence_row = company_confidence.filter(col("CompanyCode") == company_code).first()
    if confidence_row is None:
        return f"[NO DATA] {agent_result}"
    
    conf = confidence_row.confidence
    count_n = confidence_row.record_count
    
    prefix = f"[CONFIDENCE: {conf} | Based on {count_n:,} records]"
    
    # Apply output guardrail
    guard_result = output_guardrail(agent_result)
    if not guard_result["safe"]:
        return f"[BLOCKED by guardrail: {guard_result['violations']}]"
    
    if not guard_result["has_hedge"] and conf == "LOW":
        return f"{prefix} [WARN: low-confidence answer lacks hedge terms] {agent_result}"
    
    return f"{prefix} {agent_result}"

# Test
example_answer = "Company 1710 may have significant cash outflow patterns due to year-end adjustments."
print(confidence_aware_answer("1710", "test question", example_answer))

low_conf_answer = "Company USC2 shows extreme anomaly with money laundering is occurring."
print("\n" + confidence_aware_answer("USC2", "test question", low_conf_answer))

[CONFIDENCE: HIGH | Based on 808,423 records] Company 1710 may have significant cash outflow patterns due to year-end adjustments.

[BLOCKED by guardrail: ['Forbidden assertion: \\bmoney\\s+laundering\\s+is\\s+occurring\\b']]
